# Gale Crater dust devil likelihood: full pipeline in one notebook

One-sol, per-pixel **heuristic likelihood** (0 to 1) of dust devil activity on the 100 m THEMIS grid. Not a calibrated probability and not a detection.

Stages, run top to bottom in one kernel:

0. Mastcam dust opacity table (`mastcam_tau_per_sol.csv`)
1. Per-pixel feature table (`gale_dust_devil_features_100m.csv` + `gale_dust_devil_meta.json`), needs rasterio and the 20 m rasters
2. Likelihood for `DEMO_SOL` (`.npz` + `rover_sol<N>_table.csv`)
3. GeoTIFF (two bands: daily mean, daily max)
4. Overlay previews on the CTX imagery and hillshade (top 10% in `#EF4444`)

Stages 0 and 1 are skipped when their output already exists (`"auto"`); set a flag to `True` to force a rebuild, `False` to skip. To model another sol, change `DEMO_SOL` and run from the config cell down: stage 2 reads the sol-specific tau from the table, and stages 3 and 4 follow the same sol. Stage 1 only needs a re-run if an input raster, the bounding box, or `DEMO_SOL` (for the rover pixel in `meta.json`) changes.

In [13]:
from pathlib import Path
import gc, json, re
import numpy as np, pandas as pd

PROC = Path("../processed_data")
OUT = Path("../gale_tiles/out_tiles")

# ---------------- which stages to run: True = always, False = never, "auto" = only if the output is missing ----------------
RUN_TAU_TABLE = "auto"
RUN_FEATURES  = "auto"
RUN_GEOTIFF   = True

# ---------------- model settings ----------------
DEMO_SOL = 800           # the sol to model; the stage-2 output lists which sols have >=40 bins
SLOPE_SCALE_DEG = 10.0   # score falls off for slopes steeper than this
DAY_LTST = (9.0, 16.0)   # window used for the daily-mean score

# ---------------- dust opacity ----------------
TAU_FILES = {"880": Path("../raw_data") / "msl-tau-880-arch-3953.txt", "440": Path("../raw_data") / "msl-tau-440-arch-3953.txt"}   # the two Mastcam tables, used directly (no zip, no folder)
TAU_BAND = "880"         # "880", "440", or "mean" (average of the two bands, sols with both)
TAU_CSV = PROC / "mastcam_tau_per_sol.csv"   # written by stage 0, read by stage 2; set to None to use the placeholder TAU
MAX_TAU_GAP_SOLS = 5     # only Mastcam measurements within this many sols of DEMO_SOL are used; stage 2 fails if there is none
TAU = 0.5                # placeholder, used only when TAU_CSV is None
assert TAU_BAND in ("880", "440", "mean"), TAU_BAND

# ---------------- input and output files ----------------
BINNED_CSV = PROC / "rems_binned_all_sols.csv"
TEMPLATE_TIF = PROC / "gale_ti_night_themis100m_native.tif"          # output grid (100 m THEMIS thermal inertia)
DEM_TIF, SLOPE_TIF, ASPECT_TIF = PROC/"gale_dem_20m.tif", PROC/"gale_slope_20m.tif", PROC/"gale_aspect_20m.tif"
ALB_TIFS = {y: PROC/f"gale_albedo_my{y}_native.tif" for y in (24, 25, 26)}
MIN_TI_COVERAGE = 0.99   # fail stage 1 if THEMIS covers less of the template

FEATURES_CSV = PROC / "gale_dust_devil_features_100m.csv"      # one row per 100 m pixel
META_JSON = PROC / "gale_dust_devil_meta.json"                 # template grid (shape + transform) and rover pixel
NPZ = PROC / f"gale_dust_devil_likelihood_sol{DEMO_SOL}.npz"
GEOTIFF = PROC / f"gale_dust_devil_likelihood_sol{DEMO_SOL}.tif"

# ---------------- overlay previews (stage 4) ----------------
IMAGERY_PREVIEW   = OUT / "imagery_preview_z11.png"        # SET THIS: zoom-11 CTX imagery preview (.png/.jpg)
HILLSHADE_PREVIEW = OUT / "topo_preview_z11.png"   # SET THIS: zoom-11 hillshade preview (.png/.jpg)
PREVIEW_BOUNDS = None    # (west, east, south, north) in degrees if the previews cover MORE than the template; None = same as template
LIKELY_TOP_PCT = 10      # highlight the top N% of pixels by daily-mean score (relative score, not a probability)
HIGHLIGHT = "#EF4444"
ALPHA = 0.55
PANEL_W = 1100           # width of each panel in the side-by-side image, in pixels
SEPARATE_W = 2000        # width of each separate preview image, in pixels

def want(flag, output):
    assert flag in (True, False, "auto"), f"flag must be True, False or 'auto', got {flag!r}"
    return (not Path(output).exists()) if flag == "auto" else flag

## Shared solar geometry (used by stages 1 and 2)

In [2]:
# ---- Solar geometry (Allison & McEwen 2000 style; good to ~0.1 deg) ----
SOL_DAYS = 1.0274912517
LANDING_JD_TT = 2456145.7 + 69.184/86400     # MSL landing 2012-08-06, sol 0 (approx.)
def ls_deg(sol):
    jd = LANDING_JD_TT + np.asarray(sol, float) * SOL_DAYS
    dj = jd - 2451545.0
    M = np.radians(19.3871 + 0.52402073 * dj)
    alpha = 270.3863 + 0.52403840 * dj
    pbs = (0.0071*np.cos(np.radians(0.985626*dj/2.2353+49.409)) + 0.0057*np.cos(np.radians(0.985626*dj/2.7543+168.173))
           + 0.0039*np.cos(np.radians(0.985626*dj/1.1177+191.837)) + 0.0037*np.cos(np.radians(0.985626*dj/15.7866+21.736))
           + 0.0021*np.cos(np.radians(0.985626*dj/2.1354+15.704)) + 0.0020*np.cos(np.radians(0.985626*dj/2.4694+95.528))
           + 0.0018*np.cos(np.radians(0.985626*dj/32.8493+49.095)))
    eoc = (10.691 + 3.0e-7*dj)*np.sin(M) + 0.623*np.sin(2*M) + 0.050*np.sin(3*M) + 0.005*np.sin(4*M) + 0.0005*np.sin(5*M) + pbs
    return (alpha + eoc) % 360

def sun_vector(ls, lat_deg, ltst_hr):
    """Unit vector to the Sun in local (East, North, Up). lat_deg may be an array."""
    dec = np.arcsin(np.sin(np.radians(25.19)) * np.sin(np.radians(ls)))
    lat, h = np.radians(lat_deg), np.radians((ltst_hr - 12.0) * 15.0)
    sin_alt = np.sin(lat)*np.sin(dec) + np.cos(lat)*np.cos(dec)*np.cos(h)
    cos_alt = np.sqrt(np.clip(1 - sin_alt**2, 0, 1))
    cos_alt_safe = np.where(cos_alt < 1e-6, 1e-6, cos_alt)
    east  = -np.cos(dec) * np.sin(h) / cos_alt_safe          # cos(alt)*sin(az) / cos(alt)
    north = (np.sin(dec) - sin_alt*np.sin(lat)) / (cos_alt_safe*np.cos(lat))
    return east*cos_alt, north*cos_alt, sin_alt

def solar_const(ls):
    nu = np.radians(ls - 251.0)                   # perihelion near Ls 251
    r = 1.52368 * (1 - 0.0934**2) / (1 + 0.0934*np.cos(nu))
    return 589.0 * (1.52368 / r)**2               # W/m2 at Mars distance

LS = float(ls_deg(DEMO_SOL))
print(f"Ls at landing (sol 0): {float(ls_deg(0)):.1f} deg (expect ~151); Ls at sol {DEMO_SOL}: {LS:.1f} deg; S0 = {solar_const(LS):.0f} W/m2")

Ls at landing (sol 0): 150.7 deg (expect ~151); Ls at sol 800: 228.2 deg; S0 = 707 W/m2


## Stage 0: Mastcam dust opacity per sol
Parses Lemmon's Mastcam aerosol tables into one median tau per sol. Mastcam starts at sol 33, so sols 1-32 have no value.

In [3]:
TAU_OUT = TAU_CSV if TAU_CSV is not None else PROC / "mastcam_tau_per_sol.csv"

def build_tau_table():
    def read_tau(path):
        text = path.read_text()
        declared = int(re.search(r"Lines of data in this table:\s*(\d+)", text).group(1))
        lines = text.splitlines()
        start = next(i for i, l in enumerate(lines) if "Product_ID" in l) + 1
        rows = [l.split(",") for l in lines[start:] if l.strip()]
        assert all(len(r) == 5 for r in rows), f"{path.name}: a row does not have 5 fields"
        d = pd.DataFrame({"sol": [float(r[1]) for r in rows], "ls_deg": [float(r[2]) for r in rows], "tau": [float(r[3]) for r in rows],
                          "sigma": pd.to_numeric([r[4].strip() for r in rows], errors="coerce")})     # '******' -> NaN
        assert len(d) == declared, f"{path.name}: parsed {len(d)} rows but the header declares {declared}"
        n_bad = int((d.tau < 0).sum())                       # -1 = no validated data (none in the current files)
        d = d[d.tau >= 0]
        print(f"{path.name}: {len(d)} rows, sol {d.sol.min():.1f}..{d.sol.max():.1f}, tau {d.tau.min():.3f}..{d.tau.max():.3f}, dropped {n_bad} with tau = -1, {int(d.sigma.isna().sum())} with unreadable sigma")
        return d

    def per_sol(d):
        d = d.assign(sol_int=np.floor(d.sol).astype(int))
        return d.groupby("sol_int").agg(tau=("tau", "median"), n=("tau", "size"), tau_min=("tau", "min"), tau_max=("tau", "max"), ls_deg=("ls_deg", "first"))

    t880, t440 = per_sol(read_tau(TAU_FILES["880"])), per_sol(read_tau(TAU_FILES["440"]))
    if TAU_BAND == "880":
        out = t880
    elif TAU_BAND == "440":
        out = t440
    else:
        both = t880.join(t440, lsuffix="_880", rsuffix="_440", how="inner")
        out = pd.DataFrame({"tau": (both.tau_880 + both.tau_440) / 2, "n": both.n_880 + both.n_440,
                            "tau_min": both[["tau_min_880", "tau_min_440"]].min(axis=1), "tau_max": both[["tau_max_880", "tau_max_440"]].max(axis=1),
                            "ls_deg": both.ls_deg_880})
    out = out.rename_axis("sol").reset_index()
    out["band"] = TAU_BAND
    print(f"\n{len(out)} sols with tau ({TAU_BAND}); median tau {out.tau.median():.2f}, 95th percentile {out.tau.quantile(.95):.2f}, max {out.tau.max():.2f} (sol {int(out.sol[out.tau.idxmax()])})")

    # ---- coverage against the REMS sols you have ----
    rems_sols = np.sort(pd.read_csv(BINNED_CSV).sol.unique())
    dist = np.array([np.abs(out.sol.to_numpy() - s).min() for s in rems_sols])
    print(f"REMS sols: {len(rems_sols)} | tau on the same sol: {int((dist == 0).sum())} | within 5 sols: {int((dist <= 5).sum())} | within 10: {int((dist <= 10).sum())} | before the first tau sol ({int(out.sol.min())}): {int((rems_sols < out.sol.min()).sum())}")
    gaps = np.diff(out.sol.to_numpy()); print(f"gap between measured sols: median {np.median(gaps):.0f}, 90th percentile {np.percentile(gaps, 90):.0f}, max {gaps.max()} (after sol {int(out.sol.to_numpy()[gaps.argmax()])})")
    for s in (800, 1163):
        near = out.iloc[np.abs(out.sol - s).argmin()]; print(f"sol {s}: nearest measured sol {int(near.sol)}, tau {near.tau:.3f} (n={int(near.n)}), Ls {near.ls_deg:.1f}")
    out.to_csv(TAU_OUT, index=False, float_format="%.4f")
    print("wrote", TAU_OUT)

if want(RUN_TAU_TABLE, TAU_OUT):
    build_tau_table()
else:
    print("stage 0 skipped:", TAU_OUT, "exists" if TAU_OUT.exists() else "(not rebuilt)")

stage 0 skipped: processed_data\mastcam_tau_per_sol.csv exists


## Stage 1: per-pixel feature table
Resamples DEM, slope, aspect (20 m, area average; aspect through sin/cos), the three albedo maps (bilinear) and THEMIS thermal inertia onto the 100 m template, and writes the CSV and `meta.json`. Needs rasterio; it is imported only here, and all arrays are freed when the stage ends.

In [4]:
def build_feature_table():
    import rasterio
    from rasterio.warp import reproject, Resampling

    with rasterio.open(TEMPLATE_TIF) as t:
        T_TF, T_CRS, T_SHAPE = t.transform, t.crs, (t.height, t.width)
        ti = t.read(1).astype("float32")
        if t.nodata is not None: ti[ti == t.nodata] = np.nan
    assert T_CRS.is_geographic, "template should be lon/lat"
    print("template", T_SHAPE, "pixel (deg):", T_TF.a, T_TF.e)
    print("template bounds lon %.4f..%.4f  lat %.4f..%.4f" % (T_TF.c, T_TF.c + T_TF.a*T_SHAPE[1], T_TF.f + T_TF.e*T_SHAPE[0], T_TF.f))
    ti_cov = np.isfinite(ti).mean()
    assert ti_cov >= MIN_TI_COVERAGE, f"THEMIS covers only {ti_cov:.1%} of the template"
    print(f"THEMIS thermal inertia coverage: {ti_cov:.2%}")

    def to_template(path, how=Resampling.average, band_fn=None):
        with rasterio.open(path) as s:
            a = s.read(1).astype("float32")
            if s.nodata is not None: a[a == s.nodata] = np.nan
            if band_fn is not None: a = band_fn(a)
            out = np.full(T_SHAPE, np.nan, "float32")
            reproject(a, out, src_transform=s.transform, src_crs=s.crs, dst_transform=T_TF, dst_crs=T_CRS,
                      resampling=how, src_nodata=np.nan, dst_nodata=np.nan)
        return out

    dem   = to_template(DEM_TIF)
    slope = to_template(SLOPE_TIF)
    asp_s = to_template(ASPECT_TIF, band_fn=lambda a: np.sin(np.radians(a)))   # average aspect as sin/cos, never raw degrees
    asp_c = to_template(ASPECT_TIF, band_fn=lambda a: np.cos(np.radians(a)))
    aspect = np.degrees(np.arctan2(asp_s, asp_c)) % 360
    alb = {y: to_template(p, Resampling.bilinear) for y, p in ALB_TIFS.items()}
    alb_mean = np.nanmean(np.stack(list(alb.values())), axis=0)

    layers = dict(elev_m=dem, slope_deg=slope, aspect_deg=aspect, ti_themis_tiu=ti,
                  albedo_my24=alb[24], albedo_my25=alb[25], albedo_my26=alb[26], albedo_mean=alb_mean)
    for n, a in layers.items():
        print(f"{n:14s} min {np.nanmin(a):9.3f}  max {np.nanmax(a):9.3f}  NaN {100*np.mean(~np.isfinite(a)):5.1f}%")

    rows, cols = np.meshgrid(np.arange(T_SHAPE[0]), np.arange(T_SHAPE[1]), indexing="ij")
    lat = T_TF.f + (rows + 0.5) * T_TF.e
    lon = T_TF.c + (cols + 0.5) * T_TF.a
    df = pd.DataFrame({"pixel_id": (rows * T_SHAPE[1] + cols).ravel().astype("int32"),
                       "row": rows.ravel().astype("int32"), "col": cols.ravel().astype("int32"),
                       "lon": lon.ravel(), "lat": lat.ravel(),
                       **{n: a.ravel() for n, a in layers.items()}})
    n_all = len(df)
    df = df.dropna(subset=list(layers)).reset_index(drop=True)
    print(f"kept {len(df):,} of {n_all:,} pixels ({len(df)/n_all:.1%}); dropped {n_all-len(df):,} with a missing layer")
    assert len(df) > 0.9 * n_all, "more than 10% of pixels lack a layer: check that the 20 m rasters cover the template"
    df = df.round({"lon": 6, "lat": 6, "elev_m": 2, "slope_deg": 3, "aspect_deg": 2, "ti_themis_tiu": 2,
                   "albedo_my24": 5, "albedo_my25": 5, "albedo_my26": 5, "albedo_mean": 5})   # %.5g would cut lon/lat to 0.01 deg
    df.to_csv(FEATURES_CSV, index=False)
    print("wrote", FEATURES_CSV, f"({FEATURES_CSV.stat().st_size/1e6:.0f} MB)")

    rems = pd.read_csv(BINNED_CSV)
    day = rems[rems.sol == DEMO_SOL].sort_values("bin").reset_index(drop=True)
    assert len(day) > 0, f"sol {DEMO_SOL} not in {BINNED_CSV.name}"
    assert day.latitude_deg.nunique() == 1 and day.longitude_deg_e.nunique() == 1, "rover moved during the demo sol"
    c, r = ~T_TF * (day.longitude_deg_e[0], day.latitude_deg[0]); r, c = int(r), int(c)   # affine returns (col, row)
    assert 0 <= r < T_SHAPE[0] and 0 <= c < T_SHAPE[1], "rover pixel is outside the template"

    meta = dict(demo_sol=DEMO_SOL, ls_deg=float(ls_deg(DEMO_SOL)), rover_lat=float(day.latitude_deg[0]), rover_lon=float(day.longitude_deg_e[0]), rover_elev_m=float(day.elevation_m[0]),
                rover_row=r, rover_col=c, rover_pixel_id=int(r*T_SHAPE[1]+c),
                n_rems_bins=int(len(day)), template_shape=list(T_SHAPE), template_transform=list(T_TF)[:6])
    META_JSON.write_text(json.dumps(meta, indent=2))
    print(json.dumps(meta, indent=2))

    # sanity: DEM at the rover pixel vs the elevation REMS reports for the rover
    dem_rover = float(dem[r, c])
    print(f"DEM at rover pixel {dem_rover:.1f} m vs REMS elevation {meta['rover_elev_m']:.1f} m (diff {dem_rover-meta['rover_elev_m']:+.1f} m)")
    print("rover-pixel row in the CSV:", int((df.pixel_id == meta["rover_pixel_id"]).sum()), "(expect 1)")

if want(RUN_FEATURES, FEATURES_CSV):
    build_feature_table()
    gc.collect()
else:
    print("stage 1 skipped:", FEATURES_CSV, "exists" if FEATURES_CSV.exists() else "(not rebuilt)")

stage 1 skipped: processed_data\gale_dust_devil_features_100m.csv exists


## Stage 2: likelihood for `DEMO_SOL`
Per 15-minute REMS bin: absorbed insolation on the tilted surface divided by the square root of thermal inertia, scaled by its own 99.5th percentile, capped at 1, times a slope penalty, then averaged over the day window. The rover-calibrated kelvin scale cancels in the percentile, so it only affects the rover check. All weights are assumptions, not fitted.

In [5]:
d = pd.read_csv(BINNED_CSV)
per_sol = d.groupby("sol").agg(bins=("bin","size"), lat=("latitude_deg","first"), lon=("longitude_deg_e","first"),
                               elev=("elevation_m","first"))
print(f"{len(per_sol)} sols in file, sol {per_sol.index.min()}..{per_sol.index.max()}")
print("sols with >=40 bins (bins are 15 min, max 96/day):", per_sol.index[per_sol.bins >= 40].tolist())
day = d[d.sol == DEMO_SOL].sort_values("bin").reset_index(drop=True)
assert len(day) > 0, f"sol {DEMO_SOL} not in file"
print(f"\nDemo sol {DEMO_SOL}: {len(day)} bins, rover at lat {day.latitude_deg[0]:.5f}, lon {day.longitude_deg_e[0]:.5f}, elev {day.elevation_m[0]:.1f} m")
day["gap_obs_k"] = day.ground_temp_k - day.air_temp_k
day[["ltst_bin_center_hr","air_temp_k","ground_temp_k","gap_obs_k","pressure_pa"]].head(8)

272 sols in file, sol 1..3778
sols with >=40 bins (bins are 15 min, max 96/day): [21, 22, 23, 26, 30, 33, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 90, 95, 100, 105, 110, 120, 121, 122, 123, 124, 130, 135, 155, 160, 165, 170, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 584, 585, 590, 592, 610, 615, 620, 625, 630, 635, 640, 645, 650, 665, 709, 710, 720, 725, 730, 735, 740, 745, 760, 765, 799, 800, 801, 802, 803, 804, 815, 820, 825, 830, 835, 840, 845, 850, 855, 865, 1161, 1162, 1163, 1343, 1347, 1348, 1349, 1350, 1352, 1355, 1356, 1357, 1358, 1359, 1360, 1650, 1655, 1660, 1665, 1670, 1675, 1680, 1690, 1695, 1705, 1710, 1715, 1720, 1730, 1735, 1740, 1750, 1770, 1775, 1780, 1785, 1790, 1795, 1800, 1805, 1810, 1815, 1825, 1830, 1840, 1845, 1850, 1860, 1865, 3069, 3070, 3071, 3072, 3073, 3074, 3085, 3090, 3095, 3100, 3110, 3550, 3555, 3560, 3565, 3570, 3575, 35

,ltst_bin_center_hr,air_temp_k,ground_temp_k,gap_obs_k,pressure_pa
0,0.38,216.72,209.20,-7.52,914.57
1,0.62,216.72,209.93,-6.79,915.08
2,1.38,216.43,207.11,-9.32,915.60
3,1.62,216.28,208.20,-8.08,915.41
4,2.38,214.42,205.55,-8.87,913.27
5,2.62,213.69,205.51,-8.18,913.41
6,3.38,210.31,204.64,-5.67,910.79
7,3.62,209.51,203.29,-6.22,910.46


In [6]:
meta = json.loads(META_JSON.read_text())
T_SHAPE = tuple(meta["template_shape"])
T_A, T_B, T_C, T_D, T_E, T_F = meta["template_transform"]      # lon = T_C + col*T_A ; lat = T_F + row*T_E (north-up, so T_B = T_D = 0)
assert T_B == 0 and T_D == 0 and T_E < 0, "template is not a north-up lon/lat grid"
print("template", T_SHAPE, "pixel (deg):", T_A, T_E)

feat = pd.read_csv(FEATURES_CSV)
assert feat.pixel_id.is_unique, "duplicate pixel_id in the features CSV"
assert (feat.pixel_id == feat.row * T_SHAPE[1] + feat.col).all(), "pixel_id does not match row/col for this template shape"
assert feat.row.between(0, T_SHAPE[0]-1).all() and feat.col.between(0, T_SHAPE[1]-1).all(), "row/col outside the template"
print(f"features CSV: {len(feat):,} of {T_SHAPE[0]*T_SHAPE[1]:,} pixels ({len(feat)/(T_SHAPE[0]*T_SHAPE[1]):.1%}); missing pixels become NaN")

def grid(column):
    out = np.full(T_SHAPE, np.nan, "float32")
    out[feat.row.to_numpy(), feat.col.to_numpy()] = feat[column].to_numpy("float32")
    return out

dem, slope, aspect = grid("elev_m"), grid("slope_deg"), grid("aspect_deg")
alb = grid("albedo_mean")          # mean of MY24-26, same as the original demo
ti = grid("ti_themis_tiu")         # THEMIS 100 m only; no TES fallback

rows = np.arange(T_SHAPE[0]); cols = np.arange(T_SHAPE[1])
lat = (T_F + (rows + 0.5) * T_E)[:, None] * np.ones((1, T_SHAPE[1]), "float32")   # from the transform, not the CSV's rounded lon/lat
lon = (T_C + (cols + 0.5) * T_A)[None, :] * np.ones((T_SHAPE[0], 1), "float32")
for n, a in dict(dem=dem, slope=slope, aspect=aspect, albedo=alb, ti=ti).items():
    print(f"{n:7s} min {np.nanmin(a):9.2f}  max {np.nanmax(a):9.2f}  NaN {100*np.mean(~np.isfinite(a)):.1f}%")

template (1436, 1477) pixel (deg): 0.001687051876845213 -0.0016870993139129456
features CSV: 2,120,824 of 2,120,972 pixels (100.0%); missing pixels become NaN
dem     min  -4715.40  max   1115.87  NaN 0.0%
slope   min      0.00  max     82.10  NaN 0.0%
aspect  min      0.00  max    360.00  NaN 0.0%
albedo  min      0.10  max      0.27  NaN 0.0%
ti      min    102.23  max    818.64  NaN 0.0%


In [7]:
# ---- Tau for this sol: Mastcam measurements within MAX_TAU_GAP_SOLS, interpolated in sol ----
if TAU_CSV is None:
    tau, tau_source = TAU, "placeholder"
    print("TAU is the placeholder", TAU, "- set TAU_CSV to use the Mastcam tau.")
else:
    t = pd.read_csv(TAU_CSV).sort_values("sol")
    assert t.sol.is_unique, "duplicate sols in the tau CSV"
    near = t[(t.sol - DEMO_SOL).abs() <= MAX_TAU_GAP_SOLS]
    assert len(near) > 0, f"no Mastcam tau within {MAX_TAU_GAP_SOLS} sols of sol {DEMO_SOL} (table covers sols {int(t.sol.min())}-{int(t.sol.max())}, nearest is {int((t.sol - DEMO_SOL).abs().min())} sols away)"
    tau = float(np.interp(DEMO_SOL, near.sol, near.tau))
    tau_source = f"Mastcam {t.band.iloc[0]} nm, nearest {int((near.sol - DEMO_SOL).abs().min())} sol(s) away"
    print(f"tau = {tau:.3f} from {len(near)} sol(s) with data: " + ", ".join(f"sol {int(s)}: {v:.3f}" for s, v in zip(near.sol, near.tau)))

# ---- Tilted-surface insolation for each 30-min bin ----
S0 = solar_const(LS)
sl, asr = np.radians(slope), np.radians(aspect)
nE, nN, nU = np.sin(sl)*np.sin(asr), np.sin(sl)*np.cos(asr), np.cos(sl)    # aspect = direction the slope faces, deg clockwise from N

ltst_centers = day.ltst_bin_center_hr.values
flux = np.zeros((len(ltst_centers),) + T_SHAPE, "float32")
for k, hr in enumerate(ltst_centers):
    sE, sN, sU = sun_vector(LS, lat, hr)
    cos_i = np.clip(nE*sE + nN*sN + nU*sU, 0, None)
    up = sU > 0.02
    direct = np.where(up, S0 * cos_i * np.exp(-tau / np.clip(sU, 0.05, 1)), 0)
    diffuse = np.where(up, 0.15 * S0 * sU * (1 - np.exp(-tau)) * (1 + nU) / 2, 0)   # crude sky term
    flux[k] = direct + diffuse
print("peak flux over domain (W/m2):", float(np.nanmax(flux)))

tau = 1.498 from 5 sol(s) with data: sol 796: 1.471, sol 799: 1.517, sol 801: 1.478, sol 803: 1.442, sol 804: 1.468
peak flux over domain (W/m2): 232.72732543945312


In [8]:
# ---- Heuristic likelihood. ALL weights are assumptions, not fitted. ----
# gap proxy ~ absorbed flux / sqrt(thermal inertia): strong surface heating over a low-inertia surface => big ground-air gap => convection
gap_proxy = (1 - alb)[None] * flux                                   # absorbed flux
gap_proxy /= np.sqrt(np.clip(ti, 50, None))[None]                    # in place: no extra full-size copies

# calibrate the proxy to the rover's observed ground-minus-air gap on this sol
c = (day.longitude_deg_e[0] - T_C) / T_A; r = (day.latitude_deg[0] - T_F) / T_E; r, c = int(r), int(c)   # col from lon, row from lat
assert 0 <= r < T_SHAPE[0] and 0 <= c < T_SHAPE[1], "rover pixel is outside the template"
gap_rover_proxy = gap_proxy[:, r, c]
obs = day.gap_obs_k.values
ok = np.isfinite(obs) & (gap_rover_proxy > 0)
assert ok.any(), "no usable bins at the rover pixel"
scale = float(np.nanmax(obs[ok]) / np.nanmax(gap_rover_proxy[ok]))
print(f"rover pixel (row {r}, col {c}), pixel_id {r*T_SHAPE[1]+c}; meta.json says row {meta['rover_row']}, col {meta['rover_col']}")
print(f"DEM at rover pixel {dem[r, c]:.1f} m vs REMS elevation {day.elevation_m[0]:.1f} m")
print(f"proxy -> kelvin scale {scale:.3f}")
x, y = gap_rover_proxy[ok].astype("float64"), obs[ok]
print("correlation of modeled vs observed gap through the day at the rover pixel: %.2f" % (np.mean((x - x.mean()) * (y - y.mean())) / (x.std() * y.std())))   # no np.corrcoef: it calls BLAS

gap_proxy *= np.float32(scale)                                       # now kelvin; float32 on purpose (a float64 scale would double the memory)
gap_k = gap_proxy
gap_rover_model = gap_k[:, r, c].copy()
flat = np.exp(-(np.nan_to_num(slope, nan=90) / SLOPE_SCALE_DEG)**2).astype("float32")   # steep slopes suppress vortices
gap_ref = np.float32(np.nanpercentile(gap_k, 99.5))
score_t = gap_k / gap_ref
np.clip(score_t, 0, 1, out=score_t)
score_t *= flat[None]                                                # per bin, 0-1
del gap_k, gap_proxy

dm = (ltst_centers >= DAY_LTST[0]) & (ltst_centers <= DAY_LTST[1])
likelihood_mean = score_t[dm].mean(axis=0)
likelihood_max = score_t.max(axis=0)
likelihood_mean[~np.isfinite(ti)] = np.nan
print("score range (daily mean):", np.nanmin(likelihood_mean), np.nanmax(likelihood_mean))

rover pixel (row 304, col 372), pixel_id 449380; meta.json says row 304, col 372
DEM at rover pixel -4437.0 m vs REMS elevation -4461.2 m
proxy -> kelvin scale 3.475
correlation of modeled vs observed gap through the day at the rover pixel: 0.97
score range (daily mean): 1.4185315e-30 0.86833054


In [9]:
np.savez_compressed(NPZ, likelihood_mean=likelihood_mean.astype("float32"), likelihood_max=likelihood_max.astype("float32"),
                    sol=DEMO_SOL, ls_deg=LS, tau=tau, tau_source=tau_source, day_ltst=np.array(DAY_LTST), rover_row=r, rover_col=c)
print("wrote", NPZ)

# per-bin table at the rover pixel (join-ready: sol, bin, LTST, Ls, tau, observed + modeled)
tbl = day[["sol","bin","ltst_bin_center_hr","air_temp_k","ground_temp_k","pressure_pa","gap_obs_k"]].copy()
tbl["ls_deg"], tbl["tau"] = LS, tau
tbl["gap_model_k_rover"] = gap_rover_model; tbl["score_rover"] = score_t[:, r, c]
tbl.to_csv(PROC / f"rover_sol{DEMO_SOL}_table.csv", index=False)
tbl.head()

# free the large arrays before the later stages
del flux, score_t, flat, feat, dem, slope, aspect, alb, ti, lat, lon, sl, asr, nE, nN, nU
gc.collect()

wrote processed_data\gale_dust_devil_likelihood_sol800.npz


0

## Stage 3: GeoTIFF
Two bands (1 = daily-mean likelihood, 2 = maximum over the day) on the template's CRS and transform, for tiling into the web map. Needs rasterio.

In [10]:
def write_geotiff():
    import rasterio
    meta = json.loads(META_JSON.read_text())
    z = np.load(NPZ)
    day_ltst = z["day_ltst"]
    with rasterio.open(TEMPLATE_TIF) as t:
        crs, tf, shape = t.crs, t.transform, (t.height, t.width)
    assert shape == tuple(meta["template_shape"]) == z["likelihood_mean"].shape, "template, meta.json and npz come from different grids"
    assert np.allclose(list(tf)[:6], meta["template_transform"]), "meta.json transform differs from the template tif"
    assert crs.is_geographic, "template should be lon/lat"

    profile = dict(driver="GTiff", dtype="float32", count=2, height=shape[0], width=shape[1], crs=crs, transform=tf, nodata=np.nan)
    with rasterio.open(GEOTIFF, "w", **profile) as dst:
        dst.write(z["likelihood_mean"], 1); dst.set_band_description(1, f"likelihood_mean_{day_ltst[0]:g}-{day_ltst[1]:g}LTST")
        dst.write(z["likelihood_max"], 2);  dst.set_band_description(2, "likelihood_max_over_day")
    print("wrote", GEOTIFF)

if RUN_GEOTIFF:
    write_geotiff()
else:
    print("stage 3 skipped")

wrote processed_data\gale_dust_devil_likelihood_sol800.tif


## Stage 4: overlay previews
Top-N% of pixels by score highlighted on the zoom-11 CTX imagery and hillshade previews, side by side and as separate images. The cyan ring is the rover. Set `IMAGERY_PREVIEW` and `HILLSHADE_PREVIEW` in the config cell to your real files.

In [11]:
from PIL import Image, ImageDraw, ImageFont
Image.MAX_IMAGE_PIXELS = None

meta = json.loads(META_JSON.read_text())
z = np.load(PROC / f"gale_dust_devil_likelihood_sol{DEMO_SOL}.npz")
likelihood_mean = z["likelihood_mean"]; tau = float(z["tau"]); tau_source = str(z["tau_source"]); day_ltst = tuple(z["day_ltst"]); r, c = int(z["rover_row"]), int(z["rover_col"])
H, W = likelihood_mean.shape
assert [H, W] == meta["template_shape"], "npz and meta.json come from different grids"
a, _, x0, _, e, y0 = meta["template_transform"]
west, east, south, north = x0, x0 + a*W, y0 + e*H, y0
rover_lon, rover_lat = meta["rover_lon"], meta["rover_lat"]
assert abs((x0 + (c + 0.5)*a) - rover_lon) < a and abs((y0 + (r + 0.5)*e) - rover_lat) < abs(e), "rover pixel in the npz does not contain the rover"

# model vs the rover's observed ground-air gap (numbers instead of a plot)
tbl = pd.read_csv(PROC / f"rover_sol{DEMO_SOL}_table.csv")
ok = tbl.gap_obs_k.notna()
x, y = tbl.gap_model_k_rover[ok].to_numpy(float), tbl.gap_obs_k[ok].to_numpy(float)
print("model vs observed gap at the rover: corr %.2f | peak observed %.1f K at %.2f h | peak model %.1f K at %.2f h" % (
    np.mean((x-x.mean())*(y-y.mean()))/(x.std()*y.std()), y.max(), tbl.ltst_bin_center_hr[ok].iloc[y.argmax()], x.max(), tbl.ltst_bin_center_hr[ok].iloc[x.argmax()]))
print("score range (daily mean): %.4f .. %.4f" % (np.nanmin(likelihood_mean), np.nanmax(likelihood_mean)))

model vs observed gap at the rover: corr 0.98 | peak observed 28.0 K at 12.38 h | peak model 28.0 K at 12.12 h
score range (daily mean): 0.0000 .. 0.8683


In [14]:
PH = round(PANEL_W * H / W)
rgb = tuple(int(HIGHLIGHT[i:i+2], 16) for i in (1, 3, 5))

thr = np.nanpercentile(likelihood_mean, 100 - LIKELY_TOP_PCT)
likely = np.isfinite(likelihood_mean) & (likelihood_mean >= thr)
def make_mask(w):
    return np.asarray(Image.fromarray(likely.astype("uint8") * 255).resize((w, round(w * H / W)), Image.NEAREST), dtype="float32")[..., None] / 255
mask = make_mask(PANEL_W)
print(f"top {LIKELY_TOP_PCT}% threshold: score >= {thr:.3f} ({int(likely.sum()):,} pixels)")

def panel(path, name, w=PANEL_W):
    ph = round(w * H / W); m = mask if w == PANEL_W else make_mask(w)
    assert path.exists(), f"set the preview path in the config cell; not found: {path}"
    assert path.suffix.lower() in (".png", ".jpg", ".jpeg"), f"{path.name}: export the preview as .png or .jpg"
    img = Image.open(path).convert("RGB")
    if PREVIEW_BOUNDS is not None:                                   # crop the preview to the template extent
        bw, be, bs, bn = PREVIEW_BOUNDS
        assert bw <= west and be >= east and bs <= south and bn >= north, "PREVIEW_BOUNDS must contain the template"
        sx, sy = img.width / (be - bw), img.height / (bn - bs)
        img = img.crop((round((west-bw)*sx), round((bn-north)*sy), round((east-bw)*sx), round((bn-south)*sy)))
    print(f"{name}: {path.name}, {img.width}x{img.height} px; aspect {img.width/img.height:.3f} vs template {W/H:.3f}")
    base = np.asarray(img.resize((w, ph), Image.LANCZOS), dtype="float32")
    out = base * (1 - ALPHA * m) + np.array(rgb, "float32") * (ALPHA * m)
    im = Image.fromarray(out.round().astype("uint8"))
    d = ImageDraw.Draw(im)
    px, py = (rover_lon - west) / (east - west) * w, (north - rover_lat) / (north - south) * ph
    rr_ = max(10, w // 110)
    d.ellipse([px-rr_, py-rr_, px+rr_, py+rr_], outline=(0, 0, 0), width=rr_ // 2 + 1); d.ellipse([px-rr_, py-rr_, px+rr_, py+rr_], outline=(0, 255, 255), width=max(3, rr_ // 4))
    return im

font = ImageFont.load_default()
HEAD, FOOT, GAP = 54, 40, 12
canvas = Image.new("RGB", (2*PANEL_W + GAP, HEAD + PH + FOOT), "white")
for k, (path, name) in enumerate([(IMAGERY_PREVIEW, "CTX imagery"), (HILLSHADE_PREVIEW, "Hillshade")]):
    canvas.paste(panel(path, name), (k*(PANEL_W + GAP), HEAD))
d = ImageDraw.Draw(canvas)
d.text((8, 8), f"Dust devil likelihood heuristic, sol {DEMO_SOL}, mean {day_ltst[0]:g}-{day_ltst[1]:g} LTST, tau={tau:.2f} ({tau_source}). Relative score, not a probability.", fill="black", font=font)
d.text((8, 28), f"Left: CTX imagery   Right: hillshade   |   lon {west:.3f}..{east:.3f} E, lat {south:.3f}..{north:.3f}", fill="black", font=font)
d.rectangle([8, HEAD+PH+12, 28, HEAD+PH+28], fill=rgb)
d.text((36, HEAD+PH+14), f"top {LIKELY_TOP_PCT}% heuristic score (>= {thr:.2f})", fill="black", font=font)
d.ellipse([360, HEAD+PH+10, 380, HEAD+PH+30], outline=(0, 200, 200), width=3)
d.text((388, HEAD+PH+14), f"rover, sol {DEMO_SOL}", fill="black", font=font)
out_png = PROC / f"likelihood_overlay_sol{DEMO_SOL}.png"
canvas.save(out_png)
print("saved", out_png, canvas.size)

top 10% threshold: score >= 0.567 (212,083 pixels)
CTX imagery: imagery_preview_z11.png, 7680x7168 px; aspect 1.071 vs template 1.029
Hillshade: topo_preview_z11.png, 7680x7168 px; aspect 1.071 vs template 1.029
saved processed_data\likelihood_overlay_sol800.png (2212, 1163)


In [15]:
# ---- Separate preview images: one per background, same highlight, larger and uncluttered ----
for path, name, tag in [(IMAGERY_PREVIEW, "CTX imagery", "imagery"), (HILLSHADE_PREVIEW, "Hillshade", "hillshade")]:
    im = panel(path, name, w=SEPARATE_W)
    head, foot = 56, 48
    sheet = Image.new("RGB", (im.width, head + im.height + foot), "white")
    sheet.paste(im, (0, head))
    d = ImageDraw.Draw(sheet)
    d.text((8, 8), f"{name}: dust devil likelihood heuristic, sol {DEMO_SOL}, mean {day_ltst[0]:g}-{day_ltst[1]:g} LTST, tau={tau:.2f} ({tau_source}). Relative score, not a probability.", fill="black", font=font)
    d.text((8, 30), f"lon {west:.3f}..{east:.3f} E, lat {south:.3f}..{north:.3f}", fill="black", font=font)
    y = head + im.height + 14
    d.rectangle([8, y, 28, y+16], fill=rgb); d.text((36, y+2), f"top {LIKELY_TOP_PCT}% heuristic score (>= {thr:.2f})", fill="black", font=font)
    d.ellipse([360, y-2, 380, y+18], outline=(0, 200, 200), width=3); d.text((388, y+2), f"rover, sol {DEMO_SOL}", fill="black", font=font)
    out = PROC / f"likelihood_overlay_sol{DEMO_SOL}_{tag}.png"
    sheet.save(out); print("saved", out, sheet.size)

CTX imagery: imagery_preview_z11.png, 7680x7168 px; aspect 1.071 vs template 1.029
saved processed_data\likelihood_overlay_sol800_imagery.png (2000, 2048)
Hillshade: topo_preview_z11.png, 7680x7168 px; aspect 1.071 vs template 1.029
saved processed_data\likelihood_overlay_sol800_hillshade.png (2000, 2048)
